# AI Text Detector Training Notebook

This notebook mirrors the pipeline in `main.py` and uses the existing PyTorch Lightning training flow from `train.py`. All runtime configuration lives in the notebook.

In [ ]:
from __future__ import annotations

import json
from collections import Counter
from dataclasses import asdict
from pathlib import Path

from sklearn.model_selection import train_test_split

from dataset_loader import DatasetConfig, load_combined_dataset
from evaluate import evaluate_model, evaluate_per_source, print_eval_report, print_source_breakdown
from inference import Detector
from train import TrainConfig, train


In [ ]:
def split_loaded_data(result, test_ratio: float, seed: int):
    indices = list(range(len(result.texts)))
    stratify = None

    source_label_groups = [
        f"{result.samples[i].source_dataset}:{result.labels[i]}"
        for i in indices
    ]
    group_counts = Counter(source_label_groups)
    if group_counts and min(group_counts.values()) >= 2:
        stratify = source_label_groups
    else:
        label_counts = Counter(result.labels)
        if label_counts and min(label_counts.values()) >= 2:
            stratify = result.labels

    try:
        train_idx, test_idx = train_test_split(
            indices,
            test_size=test_ratio,
            random_state=seed,
            stratify=stratify,
        )
    except ValueError:
        train_idx, test_idx = train_test_split(
            indices,
            test_size=test_ratio,
            random_state=seed,
            stratify=None,
        )

    train_texts = [result.texts[i] for i in train_idx]
    train_labels = [result.labels[i] for i in train_idx]
    test_texts = [result.texts[i] for i in test_idx]
    test_labels = [result.labels[i] for i in test_idx]
    test_samples = [result.samples[i] for i in test_idx]
    return train_texts, train_labels, test_texts, test_labels, test_samples


def save_test_split(test_samples, output_path: str | Path):
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    with output_path.open("w", encoding="utf-8") as handle:
        for sample in test_samples:
            handle.write(
                json.dumps(
                    {
                        "text": sample.text,
                        "label": int(sample.label),
                        "source_dataset": sample.source_dataset,
                        "generator": sample.generator,
                        "domain": sample.domain,
                    },
                    ensure_ascii=False,
                )
                + "\n"
            )


In [ ]:
# Notebook configuration
# Supported sources:
# raid, ai_pile, gpt_wiki, human_vs_ai, ai_human_mixed, hc3,
# coat, ruatd, daigt_proper, daigt_v2, m_daigt, gsingh_train

artifact_dir = Path("checkpoints/notebook_run")
test_ratio = 0.15

source_paths = {
    # Example:
    # "daigt_proper": "/absolute/path/to/train_v2_drcat_02.csv",
    # "m_daigt": "/absolute/path/to/m_daigt_directory",
}

data_config = DatasetConfig(
    sources=["hc3", "gpt_wiki"],
    source_paths=source_paths,
    max_per_source=100_000,
    max_total=200_000,
    min_text_length=35,
    max_text_length=3_000,
    balance_labels=True,
    balance_sources=False,
    seed=42,
    cache_dir=".cache/datasets",
    auto_download_kaggle=True,
    cache_sources=True,
)

train_config = TrainConfig(
    d_model=256,
    nhead=8,
    num_layers=4,
    dim_feedforward=512,
    max_len=512,
    dropout=0.1,
    epochs=10,
    batch_size=32,
    lr=3e-4,
    weight_decay=0.01,
    warmup_steps=100,
    max_vocab=50_000,
    val_split=0.1,
    seed=42,
    use_amp=True,
    token_dropout=0.05,
    label_smoothing=0.02,
    patience=3,
    grad_clip_val=1.0,
    accumulate_grad_batches=1,
    num_workers=-1,
    save_dir=str(artifact_dir),
)

live_examples = [
    ("Ну блин, опять забыл зонт и промок как собака.", "Human"),
    ("Just grabbed tacos, best decision I've made all week lol", "Human"),
    ("spent 3 hours debugging only to find a missing comma", "Human"),
    (
        "The implementation of machine learning algorithms in healthcare has demonstrated significant potential for improving diagnostic accuracy.",
        "AI",
    ),
    (
        "In conclusion, the comprehensive analysis reveals a multifaceted landscape that necessitates careful consideration.",
        "AI",
    ),
    (
        "Furthermore, it is important to note that the integration of these systems requires a holistic understanding of the underlying frameworks.",
        "AI",
    ),
]

print("Artifacts:", artifact_dir)
print("Sources:", data_config.sources)
print("Epochs:", train_config.epochs)
print("Batch size:", train_config.batch_size)


## Step 1. Load Combined Dataset

In [ ]:
result = load_combined_dataset(data_config)
train_texts, train_labels, test_texts, test_labels, test_samples = split_loaded_data(
    result,
    test_ratio=test_ratio,
    seed=train_config.seed,
)

print(f"Total samples: {len(result.texts):,}")
print(f"Train samples: {len(train_texts):,}")
print(f"Test samples: {len(test_texts):,}")
print("Train labels:", Counter(train_labels))
print("Test labels:", Counter(test_labels))
print("Test sources:", Counter(sample.source_dataset for sample in test_samples))


## Step 2. Train With PyTorch Lightning

In [ ]:
model, tokenizer, threshold = train(train_texts, train_labels, train_config)

artifact_dir.mkdir(parents=True, exist_ok=True)
save_test_split(test_samples, artifact_dir / "test_samples.jsonl")

run_config = {
    "artifact_dir": str(artifact_dir),
    "test_ratio": test_ratio,
    "data_config": asdict(data_config),
    "train_config": asdict(train_config),
}
(artifact_dir / "notebook_run_config.json").write_text(
    json.dumps(run_config, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

print(f"Saved checkpoint directory: {artifact_dir}")
print(f"Saved test split: {artifact_dir / 'test_samples.jsonl'}")
print(f"Saved notebook config: {artifact_dir / 'notebook_run_config.json'}")
print(f"Calibrated threshold: {threshold:.3f}")


## Step 3. Evaluate On Held-Out Test Set

In [ ]:
eval_result = evaluate_model(
    model,
    tokenizer,
    test_texts,
    test_labels,
    batch_size=train_config.batch_size,
    threshold=threshold,
)
print_eval_report(eval_result, title="Test Set Evaluation")


## Step 4. Per-Source Breakdown

In [ ]:
detector = Detector(model, tokenizer, threshold=threshold)
breakdowns = evaluate_per_source(detector, test_samples)
if breakdowns:
    print_source_breakdown(breakdowns)
else:
    print("Per-source breakdown is unavailable for the current test split.")


## Step 5. Live Inference Examples

In [ ]:
correct = 0
for text, expected in live_examples:
    prediction = detector.predict(text)
    ok = prediction.label == expected
    correct += int(ok)
    print(
        {
            "ok": ok,
            "expected": expected,
            "predicted": prediction.label,
            "confidence": round(prediction.confidence, 4),
            "text": text,
        }
    )

print(f"Live accuracy: {correct}/{len(live_examples)}")
print(f"Checkpoint path: {artifact_dir}")
print(f"Threshold: {threshold:.3f}")
print("Usage:")
print("  from inference import Detector")
print(f"  detector = Detector.from_checkpoint('{artifact_dir.as_posix()}/')")
print("  result = detector.predict('some text')")
print("  print(result.label, result.confidence)")
